# Capítulo 7 · Entender antes de cambiar

El agente del capítulo 6 (contrato, retroalimentación, puerta de verificación y revisor con rúbrica) con una fase previa: antes de cambiar nada, explora el repositorio y escribe el marco de la tarea, con hechos y su evidencia, y un plan con una comprobación por paso. Y una forma de medir: repetir cada variante con muestreo y semillas distintas, porque una sola ejecución es una muestra.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo, el repositorio y las herramientas

In [ ]:
!pip install -q transformers==5.19.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 7: enmarcar la tarea y planificar con evidencias."""
import ast
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
RAIZ = Path.cwd()                           # donde están datos/ y comun_libro/
REPOSITORIO = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
               "archive/refs/heads/main.zip")
TAREAS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
          "T17", "T18", "T19", "T20"]       # las tareas de una sola sesión
MAX_PASOS = 30             # llamadas al modelo por tarea
MAX_TOKENS_PASO = 600      # tokens nuevos por llamada
LIMITE_ENTRADA = 12_000    # tokens de la conversación: más no cabe en la T4
RESULTADOS_INTACTOS = 2    # los últimos resultados de herramientas no se recortan
MAX_RECHAZOS = 3           # veces que la puerta devuelve el trabajo
VORAZ = {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}
MUESTREO = {"do_sample": True, "temperature": 0.7, "top_p": 0.8,
            "top_k": 20}           # los valores que recomienda Qwen para este modelo
decodificacion = dict(VORAZ)       # la que usa generar(); probar() la cambia
hr = None                  # el módulo de herramientas, tras preparar()



def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def preparar():
    """El repositorio de Ciclos Norte, sus tareas y las herramientas del agente.
    Si no están (en Colab), se descargan del repositorio del libro."""
    global hr
    if not (RAIZ / "comun_libro" / "herramientas_repo.py").exists():
        with urllib.request.urlopen(REPOSITORIO) as respuesta:
            archivo = zipfile.ZipFile(io.BytesIO(respuesta.read()))
        for nombre in archivo.namelist():
            relativa = nombre.split("/", 1)[1]
            if relativa.startswith(("datos/", "comun_libro/")) and not nombre.endswith("/"):
                destino = RAIZ / relativa
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(archivo.read(nombre))
    sys.path.insert(0, str(RAIZ / "comun_libro"))
    hr = importlib.import_module("herramientas_repo")
    print(f"Repositorio en {hr.REPO}: {len(hr.archivos_de(hr.REPO))} archivos")

In [ ]:
tok, modelo = cargar()
preparar()

## 2. El modelo

`generar` usa la decodificación que diga `decodificacion`: voraz o con muestreo.

In [ ]:
def plantilla(tok, mensajes):
    """La conversación como texto, con las herramientas, en la plantilla del modelo."""
    return tok.apply_chat_template(mensajes, tools=hr.ESQUEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generar(tok, modelo, texto):
    """Una respuesta del modelo, con la decodificación del momento: voraz o con
    muestreo. Devuelve el texto y los tokens de entrada y de salida."""
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, **decodificacion, max_new_tokens=MAX_TOKENS_PASO,
                                 prefill_chunk_size=1024, pad_token_id=tok.eos_token_id)
    n_entrada = entrada["input_ids"].shape[1]
    nuevos = salida[0, n_entrada:].cpu()
    del entrada, salida
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # cada paso tiene otra longitud: sin esto la
        #                           memoria reservada crece hasta llenar la GPU
    return tok.decode(nuevos, skip_special_tokens=True).strip(), n_entrada, len(nuevos)


def leer_llamadas(respuesta):
    """Las llamadas {name, arguments} que el modelo escribe entre <tool_call> y
    </tool_call>. Una llamada que no se entiende queda con name None."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", respuesta, re.S):
        try:
            datos = json.loads(bloque.strip(), strict=False)
        except ValueError:
            datos = None
        if isinstance(datos, dict) and "name" in datos:
            argumentos = datos.get("arguments", {})
            llamadas.append({"name": datos["name"],
                             "arguments": argumentos if isinstance(argumentos, dict)
                             else {}})
        else:
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def recortar_conversacion(tok, mensajes):
    """Si la conversación pasa de LIMITE_ENTRADA tokens, cambia los resultados de
    herramientas más antiguos por un aviso corto. Devuelve el texto para el modelo y
    cuántos ha recortado (None si ni así cabe)."""
    texto = plantilla(tok, mensajes)
    recortados = 0
    candidatos = [m for m in mensajes if m["role"] == "tool"
                  and not m.get("recortado")][:-RESULTADOS_INTACTOS or None]
    while len(tok(texto)["input_ids"]) > LIMITE_ENTRADA:
        if not candidatos:
            return texto, None
        mensaje = candidatos.pop(0)
        mensaje["content"] = json.dumps(
            {"aviso": "Resultado antiguo recortado para ahorrar memoria; vuelve a "
                      "pedirlo si lo necesitas.",
             "inicio": mensaje["content"][:150]}, ensure_ascii=False)
        mensaje["recortado"] = True
        recortados += 1
        texto = plantilla(tok, mensajes)
    return texto, recortados

## 3. Las reglas del capítulo 2 y el contrato del capítulo 4

In [ ]:
SISTEMA = (
    "Eres un agente de programación que trabaja en el repositorio de Ciclos Norte, "
    "una tienda de bicicletas de Oviedo. Es un programa de gestión en Python con "
    "varios módulos (pedidos, devoluciones, existencias, facturas, avisos…), sus "
    "pruebas en la carpeta tests/ y unos datos de ejemplo. Recibirás una petición de "
    "alguien de la tienda y tienes que resolverla cambiando el código con las "
    "herramientas: listar archivos, leer por rangos de líneas, buscar, editar por "
    "sustitución exacta, ejecutar las pruebas y unas pocas órdenes permitidas. "
    "Trabaja paso a paso, con una herramienta cada vez. Cuando hayas terminado, "
    "responde sin llamar a ninguna herramienta, empezando por «TERMINADO:» y "
    "explicando en una o dos frases qué has cambiado. Si no puedes terminar, empieza "
    "por «NO TERMINADO:» y explica por qué.")


AGENTS_REGLAS = """# AGENTS.md · Ciclos Norte

Reglas del repositorio. Cada una se comprueba al final de tu trabajo.

R1. Antes de editar un archivo, léelo con leer_archivo.
R2. No cambies nada de datos/.
R3. Cambia docs/ solo si la petición pide cambiar las condiciones generales.
R4. No borres ni cambies pruebas que ya existen. Excepción: si la petición cambia una
    regla del negocio, actualiza las pruebas de la regla antigua y dilo en tu respuesta.
R5. Añade tus pruebas en el archivo del módulo (tests/test_envios.py para envios.py):
    sustituye la línea `if __name__ == "__main__":` por una clase nueva con tu prueba,
    seguida de esa misma línea. Importa dentro de la prueba lo que necesites.
R6. No crees módulos nuevos fuera de tests/.
R7. «TERMINADO» exige: cambios en el código del programa, una comprobación nueva
    en tests/ y ejecutar_pruebas en verde después de tu último cambio.
R8. Tu respuesta final nombra todos los archivos que has cambiado.
"""


PIDE_DOCS = re.compile(r"(actualiz\w*|y en las)\s[^.]{0,40}condiciones\s+generales", re.I)


def lineas_del_diff(diff):
    """Las líneas añadidas y quitadas de cada archivo de un diff unificado:
    {ruta: {"+": [...], "-": [...]}}."""
    archivos, actual = {}, None
    for linea in diff.splitlines():
        if linea.startswith("+++ b/"):
            actual = archivos.setdefault(linea[6:], {"+": [], "-": []})
        elif linea.startswith("--- a/") or actual is None:
            continue
        elif linea[:1] in "+-" and linea[1:].strip():
            actual[linea[0]].append(linea[1:])
    return archivos


def comprobar_reglas(tarea, registro, cambios, respuesta, exigir_prueba=True,
                     permite_docs=None):
    """Las reglas incumplidas, como {regla: evidencia}. Mira lo que de verdad ha
    cambiado (el diff), el registro de llamadas y la respuesta final. Con
    «exigir_prueba» en False, la R7 no pide una comprobación nueva; «permite_docs»,
    si se da, decide la R3 en lugar de la petición."""
    if permite_docs is None:
        permite_docs = bool(PIDE_DOCS.search(tarea["peticion"]))
    rotas = {}
    lineas = lineas_del_diff(cambios["diff"])
    leidos = set()
    for r in registro:
        ruta = r["argumentos"].get("ruta", "")
        if r["herramienta"] == "leer_archivo" and r["ok"]:
            leidos.add(ruta)
        elif r["herramienta"] == "editar" and r["ok"] and ruta not in leidos:
            rotas.setdefault("R1", f"editaste {ruta} sin leerlo antes")
    for ruta in cambios["cambiados"]:
        if ruta.startswith("datos/"):
            rotas["R2"] = f"cambiaste {ruta}"
        if ruta.startswith("docs/") and not permite_docs:
            rotas["R3"] = f"cambiaste {ruta} y la petición no lo pide"
        if ruta.startswith("tests/") and ruta in cambios["nuevos"]:
            rotas["R5"] = f"creaste {ruta}; las pruebas van en el archivo del módulo"
        if not ruta.startswith(("tests/", "docs/", "datos/")) and ruta in cambios["nuevos"]:
            rotas["R6"] = f"creaste el módulo {ruta}"
    quitadas = [r for r, l in lineas.items() if r.startswith("tests/") and l["-"]]
    if quitadas and not re.search(r"prueba", respuesta or "", re.I):
        rotas["R4"] = (f"cambiaste pruebas que ya existían en {', '.join(quitadas)} "
                       "y tu respuesta no lo dice")
    if respuesta and dice_terminado(respuesta):
        programa = [r for r in cambios["cambiados"]
                    if not r.startswith(("tests/", "docs/", "datos/"))]
        prueba_nueva = any(re.search(r"def test_|assert", l)
                           for r, l2 in lineas.items() if r.startswith("tests/")
                           for l in l2["+"])
        ediciones = [i for i, r in enumerate(registro)
                     if r["herramienta"] in ("editar", "crear_archivo") and r["ok"]]
        despues = registro[ediciones[-1] + 1:] if ediciones else registro
        pruebas = [r for r in despues if r["herramienta"] == "ejecutar_pruebas"]
        falta = [texto for cumple, texto in (
            (programa, "no hay cambios en el código del programa"),
            (prueba_nueva or not exigir_prueba,
             "no hay ninguna comprobación nueva en tests/"),
            (pruebas and pruebas[-1]["resultado"].get("todas_pasan"),
             "no hay una ejecución de las pruebas en verde después del último cambio"))
            if not cumple]
        if falta:
            rotas["R7"] = "; ".join(falta)
        sin_nombrar = [r for r in cambios["cambiados"]
                       if Path(r).name not in respuesta]
        if sin_nombrar:
            rotas["R8"] = f"tu respuesta no nombra {', '.join(sin_nombrar)}"
    return rotas


def dice_terminado(respuesta):
    """True si la respuesta final dice que la tarea está hecha. Una respuesta vacía
    no dice nada."""
    if not respuesta.strip():
        return False
    if re.search(r"NO\s+TERMINADO", respuesta, re.I):
        return False
    if re.search(r"TERMINADO", respuesta):
        return True
    return not re.search(r"no (he podido|puedo|consigo|es posible)|necesito|\?",
                         respuesta, re.I)


class HerramientasConReglas:
    """Las herramientas del agente con las reglas de proceso comprobadas antes de
    cada edición: si la edición incumple R1, R2, R3, R5 o R6, no se hace y el agente
    recibe la regla. El resto de llamadas pasan tal cual."""

    def __init__(self, herramientas, tarea):
        self.herramientas = herramientas
        self.registro = herramientas.registro
        self.tarea = tarea
        self.leidos = set()

    def regla_rota(self, nombre, ruta):
        """La regla que incumpliría editar o crear «ruta», o None."""
        existe = self.herramientas.espacio.ruta(ruta).exists()
        if ruta.startswith("datos/"):
            return "R2: no cambies nada de datos/."
        if ruta.startswith("docs/") and not self.permite_docs():
            return "R3: la petición no pide cambiar las condiciones generales."
        if nombre == "crear_archivo" and ruta.startswith("tests/"):
            return ("R5: no crees archivos de pruebas; añade tu prueba al archivo del "
                    "módulo, sustituyendo la línea if __name__ == \"__main__\":")
        if nombre == "crear_archivo" and not existe:
            return "R6: no crees módulos nuevos fuera de tests/."
        if nombre == "editar" and existe and ruta not in self.leidos:
            return f"R1: lee {ruta} con leer_archivo antes de editarlo."
        return None

    def permite_docs(self):
        """True si la petición pide cambiar las condiciones generales (R3)."""
        return bool(PIDE_DOCS.search(self.tarea["peticion"]))

    def llamar(self, nombre, argumentos):
        """Como Herramientas.llamar, pero comprobando las reglas antes de editar."""
        ruta = str((argumentos or {}).get("ruta", ""))
        if nombre in ("editar", "crear_archivo"):
            regla = self.regla_rota(nombre, ruta)
            if regla:
                resultado = {"error": "Edición rechazada por AGENTS.md. " + regla}
                self.registro.append({"herramienta": nombre, "argumentos": argumentos,
                                      "ok": False, "segundos": 0,
                                      "resultado": resultado})
                return resultado
        resultado = self.herramientas.llamar(nombre, argumentos)
        if nombre == "leer_archivo" and "error" not in resultado:
            self.leidos.add(ruta)
        return resultado


def contrato_texto(alcance):
    """Un contrato (como datos/tareas/Txx/alcance.json) escrito para el agente:
    objetivo, rutas que puede cambiar, rutas prohibidas y criterios de aceptación
    numerados (C1, C2…)."""
    criterios = "\n".join(f"C{i}. {c}" for i, c in
                          enumerate(alcance["criterios_aceptacion"], 1))
    return ("Contrato de la tarea\n"
            f"Objetivo: {alcance['objetivo']}\n"
            f"Puedes cambiar: {', '.join(alcance['rutas_permitidas'])}\n"
            f"No toques: {', '.join(alcance['rutas_prohibidas'])}\n"
            f"Criterios de aceptación:\n{criterios}\n"
            "El contrato manda sobre AGENTS.md: si prohíbe tests/, no añadas pruebas.")


def prohibe_pruebas(alcance):
    """True si el contrato prohíbe cambiar tests/ (por ejemplo, una refactorización
    que debe pasar las pruebas sin tocarlas)."""
    return hr.coincide("tests/x.py", alcance["rutas_prohibidas"])


def comprobar_contrato(alcance, cambios, respuesta):
    """Lo que incumple el contrato al terminar: archivos fuera de las rutas permitidas
    o en las prohibidas, y criterios que la respuesta no explica cómo se cumplen."""
    rotas = {}
    rutas = hr.comprobar_alcance(cambios["cambiados"], alcance)
    fuera = sorted(set(rutas["fuera_de_alcance"]) | set(rutas["prohibidos"]))
    if fuera:
        rotas["Alcance"] = f"cambiaste {', '.join(fuera)}, que el contrato no permite"
    numeros = range(1, len(alcance["criterios_aceptacion"]) + 1)
    sin_explicar = [f"C{i}" for i in numeros
                    if not re.search(rf"\bC{i}\b", respuesta or "")]
    if sin_explicar:
        rotas["Criterios"] = ("tu respuesta no dice cómo se cumple "
                              + ", ".join(sin_explicar)
                              + " (cita cada criterio por su número)")
    return rotas


class HerramientasConContrato(HerramientasConReglas):
    """Las herramientas con las reglas del capítulo 2 y, además, el contrato: no se
    puede editar ni crear nada fuera de las rutas permitidas."""

    def __init__(self, herramientas, tarea, alcance):
        super().__init__(herramientas, tarea)
        self.alcance = alcance

    def permite_docs(self):
        """El contrato manda sobre la R3: docs/ se puede cambiar si lo permite."""
        return hr.coincide("docs/condiciones_generales.md", self.alcance["rutas_permitidas"])

    def regla_rota(self, nombre, ruta):
        """La regla o la parte del contrato que incumpliría la edición, o None."""
        alcance = self.alcance
        if hr.coincide(ruta, alcance["rutas_prohibidas"]):
            return f"Contrato: {ruta} está entre las rutas que no puedes tocar."
        if not hr.coincide(ruta, alcance["rutas_permitidas"]):
            return (f"Contrato: {ruta} no está entre las rutas que puedes cambiar ("
                    + ", ".join(alcance["rutas_permitidas"]) + ").")
        return super().regla_rota(nombre, ruta)

## 4. La retroalimentación y la puerta de verificación del capítulo 5

In [ ]:
MAX_FALLOS = 3   # fallos que se resumen para el agente


def resumir_fallos(salida, carpeta):
    """Los fallos de una ejecución de unittest, uno por prueba: qué prueba falla, las
    líneas del repositorio por las que pasa el error (sin las de Python) y la última
    línea, la del error. Las rutas se dejan relativas a la copia del repositorio."""
    fallos = []
    prefijo = str(carpeta) + "\\" if "\\" in str(carpeta) else str(carpeta) + "/"
    for bloque in re.split(r"^={50,}$", salida, flags=re.M)[1:]:
        cabecera = re.search(r"^(FAIL|ERROR): (\S+)", bloque, re.M)
        if not cabecera:
            continue
        lineas = bloque.split("-" * 70)[1] if "-" * 70 in bloque else bloque
        donde = []
        for m in re.finditer(r'File "([^"]+)", line (\d+), in (\S+)\n\s+(.+)', lineas):
            if m.group(1).startswith(prefijo):
                ruta = m.group(1)[len(prefijo):].replace("\\", "/")
                donde.append(f"{ruta}:{m.group(2)} ({m.group(3)}): {m.group(4).strip()}")
        error = [l for l in lineas.strip().splitlines() if l.strip()][-1:]
        fallos.append({"prueba": f"{cabecera.group(1)} {cabecera.group(2)}",
                       "donde": donde[-3:], "error": error[0].strip() if error else ""})
    return fallos[:MAX_FALLOS]


class HerramientasConRetro(HerramientasConContrato):
    """Las herramientas con el contrato y, además, retroalimentación: después de cada
    edición que sale bien se ejecutan las pruebas y el resultado de la edición lleva
    un resumen de los fallos. ejecutar_pruebas devuelve el mismo resumen."""

    def pruebas(self):
        """Las pruebas del repositorio, resumidas para el agente."""
        resultado = self.herramientas.ejecutar_pruebas()
        resumen = {"todas_pasan": resultado["todas_pasan"],
                   "resultado": resultado["resultado"]}
        if not resultado["todas_pasan"]:
            resumen["fallos"] = resumir_fallos(resultado["salida"],
                                               self.herramientas.espacio.carpeta)
        return resumen

    def llamar(self, nombre, argumentos):
        """Como HerramientasConContrato.llamar, con las pruebas tras cada edición."""
        if nombre == "ejecutar_pruebas":
            resultado = self.pruebas()
            self.registro.append({"herramienta": nombre, "argumentos": {}, "ok": True,
                                  "segundos": 0, "resultado": resultado})
            return resultado
        resultado = super().llamar(nombre, argumentos)
        if nombre in ("editar", "crear_archivo") and "error" not in resultado:
            resultado = dict(resultado, pruebas_tras_editar=self.pruebas())
        return resultado


def codigo_sin_docstrings(texto):
    """El árbol sintáctico de un programa, como texto, sin sus docstrings. Los
    comentarios no llegan al árbol, así que dos versiones que solo se diferencian en
    comentarios o docstrings dan el mismo resultado. None si no se puede analizar."""
    try:
        arbol = ast.parse(texto)
    except SyntaxError:
        return None
    for nodo in ast.walk(arbol):
        cuerpo = getattr(nodo, "body", None)
        if (isinstance(cuerpo, list) and cuerpo and isinstance(cuerpo[0], ast.Expr)
                and isinstance(cuerpo[0].value, ast.Constant)
                and isinstance(cuerpo[0].value.value, str)):
            nodo.body = cuerpo[1:] or [ast.Pass()]
    return ast.dump(arbol)


def cambia_el_codigo(espacio, ruta):
    """True si el archivo ha cambiado en algo más que comentarios y docstrings."""
    original, nuevo = espacio.origen / ruta, espacio.ruta(ruta)
    if not ruta.endswith(".py") or not original.exists() or not nuevo.exists():
        return True
    antes = codigo_sin_docstrings(original.read_text(encoding="utf-8"))
    despues = codigo_sin_docstrings(nuevo.read_text(encoding="utf-8"))
    return antes is None or despues is None or antes != despues


def puerta_de_verificacion(espacio, alcance):
    """Lo que falta para dar la tarea por hecha, mirando el repositorio y no lo que dice
    el agente: cambios de código que no sean solo comentarios, una prueba nueva que
    falle con el código original (si el contrato deja tocar tests/) y todas las
    pruebas en verde. Devuelve una lista de frases; vacía si se puede aceptar."""
    cambios = hr.diferencias(espacio.origen, espacio.carpeta)
    lineas = lineas_del_diff(cambios["diff"])
    falta = []
    codigo = [r for r in cambios["cambiados"]
              if not r.startswith(("tests/", "docs/", "datos/"))]
    if not any(cambia_el_codigo(espacio, r) for r in codigo):
        falta.append("no hay cambios de código en el programa (los comentarios no cuentan)")
    pruebas_nuevas = [r for r, l in lineas.items() if r.startswith("tests/")
                      and any(re.search(r"def test_|assert", x) for x in l["+"])]
    if not prohibe_pruebas(alcance):
        if not pruebas_nuevas:
            falta.append("no hay ninguna prueba nueva en tests/")
        else:
            with hr.Espacio() as original:      # las pruebas nuevas, con el código de antes
                for ruta in pruebas_nuevas:
                    destino = original.ruta(ruta)
                    destino.write_text(espacio.ruta(ruta).read_text(encoding="utf-8"),
                                       encoding="utf-8")
                antes = hr.Herramientas(original).ejecutar_pruebas()
            if antes["todas_pasan"]:
                falta.append("tus pruebas nuevas también pasan con el código original, así "
                             "que no comprueban el cambio que pide la petición")
    ahora = hr.Herramientas(espacio).ejecutar_pruebas()
    if not ahora["todas_pasan"]:
        fallos = resumir_fallos(ahora["salida"], espacio.carpeta)
        falta.append("las pruebas no pasan: "
                     + "; ".join(f"{f['prueba']} → {f['error']}" for f in fallos))
    return falta

## 5. El revisor con rúbrica del capítulo 6

In [ ]:
MAX_PASOS_REVISOR = 10   # llamadas al modelo en cada revisión


MAX_REVISIONES = 2       # veces que el revisor puede devolver el trabajo


ESCALADO = ("NO TERMINADO: el revisor ha rechazado el cambio dos veces; queda "
            "pendiente de que lo revise una persona.")


SOLO_LECTURA = ("listar_archivos", "leer_archivo", "buscar", "ejecutar_pruebas")


SISTEMA_REVISOR = (
    "Eres el revisor de cambios del repositorio de Ciclos Norte, una tienda de "
    "bicicletas de Oviedo. Otro agente ha hecho un cambio para resolver una petición y "
    "dice que ha terminado. Tu trabajo es comprobar si el cambio resuelve de verdad la "
    "petición. Puedes listar, leer y buscar archivos y ejecutar las pruebas, pero no "
    "puedes cambiar nada. Usa una herramienta cada vez. Cuando hayas terminado, "
    "responde sin llamar a ninguna herramienta, solo con un JSON.")


PIDE_LIBRE = """Revisa este cambio.

Petición:
{peticion}

Cambio (diff):
{diff}

Lo que dice el autor:
{respuesta}

Responde al final con este JSON:
{{"aprobado": true o false, "comentarios": ["…"]}}"""


PIDE_RUBRICA = """Revisa este cambio criterio por criterio.

Petición:
{peticion}

Criterios de aceptación:
{criterios}

Cambio (diff):
{diff}

Lo que dice el autor:
{respuesta}

Para cada criterio, busca en el código (no en lo que dice el autor) si se cumple, y da
la evidencia: archivo y línea, o la prueba que lo comprueba. Mira también si el cambio
toca algo que la petición no pide. Responde al final con este JSON:
{{"criterios": [{{"id": "C1", "cumple": true o false, "evidencia": "…"}}],
  "fuera_de_lo_pedido": ["…"], "aprobado": true o false}}
Aprueba solo si se cumplen todos los criterios."""


class HerramientasDeLectura:
    """Las herramientas del revisor: las de siempre, pero solo las que no cambian
    nada. Cualquier otra se rechaza."""

    def __init__(self, herramientas):
        self.herramientas = herramientas
        self.registro = []
        self.leidas = {}   # ruta -> números de línea leídos con leer_archivo

    def llamar(self, nombre, argumentos):
        """Como Herramientas.llamar, solo para las herramientas de lectura."""
        if nombre not in SOLO_LECTURA:
            resultado = {"error": f"El revisor no puede usar «{nombre}»: solo "
                                  + ", ".join(SOLO_LECTURA) + "."}
        else:
            resultado = self.herramientas.llamar(nombre, argumentos)
        self.registro.append({"herramienta": nombre, "ok": "error" not in resultado})
        rango = re.match(r"(\d+)-(\d+)", str(resultado.get("lineas", "")))
        if nombre == "leer_archivo" and rango:
            self.leidas.setdefault(resultado["ruta"], set()).update(
                range(int(rango.group(1)), int(rango.group(2)) + 1))
        return resultado


def leer_veredicto(respuesta):
    """El último objeto JSON de la respuesta del revisor, o None si no hay ninguno que
    tenga la clave «aprobado»."""
    for trozo in reversed(re.findall(r"\{.*\}", respuesta, re.S)):
        try:
            veredicto = json.loads(trozo, strict=False)
        except ValueError:
            continue
        if isinstance(veredicto, dict) and "aprobado" in veredicto:
            return veredicto
    return None


def revisar(tok, modelo, tarea, alcance, espacio, respuesta_autor, rubrica, ver=False):
    """Una revisión: un agente aparte, con su propia conversación y herramientas de
    solo lectura sobre la copia del repositorio, que devuelve su veredicto. Sin
    veredicto legible, se aprueba (el revisor no bloquea por fallar él)."""
    cambios = hr.diferencias(espacio.origen, espacio.carpeta)
    criterios = "\n".join(f"C{i}. {c}" for i, c in
                          enumerate(alcance["criterios_aceptacion"], 1))
    plantilla_peticion = PIDE_RUBRICA if rubrica else PIDE_LIBRE
    mensajes = [{"role": "system", "content": SISTEMA_REVISOR},
                {"role": "user", "content": plantilla_peticion.format(
                    peticion=tarea["peticion"], criterios=criterios,
                    diff=cambios["diff"][:3000], respuesta=respuesta_autor[:800])}]
    herramientas = HerramientasDeLectura(hr.Herramientas(espacio))
    veredicto, pasos = None, 0
    for pasos in range(1, MAX_PASOS_REVISOR + 1):
        texto, recortados = recortar_conversacion(tok, mensajes)
        if recortados is None:
            break
        respuesta, _, _ = generar(tok, modelo, texto)
        llamadas = leer_llamadas(respuesta)
        if not llamadas:
            veredicto = leer_veredicto(respuesta)
            break
        mensajes.append({"role": "assistant", "content": respuesta.split("<tool_call>")[0],
                         "tool_calls": [{"type": "function", "function": l}
                                        for l in llamadas if l["name"]]})
        for llamada in llamadas:
            resultado = (herramientas.llamar(llamada["name"], llamada["arguments"])
                         if llamada["name"] else {"error": "Llamada no válida."})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": json.dumps(resultado, ensure_ascii=False)})
    if ver:
        print(f"    REVISOR ({pasos} pasos): {json.dumps(veredicto, ensure_ascii=False)[:300]}")
    return {"aprobado": True if veredicto is None else bool(veredicto.get("aprobado")),
            "veredicto": veredicto, "pasos": pasos,
            "llamadas": len(herramientas.registro)}


def comentarios_para_el_autor(veredicto):
    """Lo que el revisor objeta, en frases para el autor."""
    if not veredicto:
        return "el revisor no aprueba el cambio"
    frases = [f"{c.get('id', '?')} no se cumple: {c.get('evidencia', '')}"
              for c in veredicto.get("criterios", []) if not c.get("cumple", True)]
    frases += [f"fuera de lo pedido: {x}" for x in veredicto.get("fuera_de_lo_pedido", [])]
    frases += [str(x) for x in veredicto.get("comentarios", [])]
    return "; ".join(frases) or "el revisor no aprueba el cambio"

## 6. El marco y el plan

La comprobación de evidencias se puede probar sin modelo: una que existe y otra que no.

In [ ]:
MAX_PASOS_MARCO = 10   # llamadas al modelo para explorar y escribir el marco


PIDE_MARCO = """Antes de cambiar nada, entiende la tarea. Explora el repositorio con las
herramientas de lectura y escribe el marco de la tarea en JSON, sin nada más al final:
{{"objetivo": "una frase",
  "hechos": [{{"hecho": "lo que has comprobado", "evidencia": "archivo:línea"}}],
  "rutas": ["archivos que habrá que cambiar"],
  "incognitas": ["lo que no sabes y cómo lo averiguarías"]}}
Cada hecho debe citar un archivo y una línea que lo demuestren.

Petición:
{peticion}

{contrato}"""


PIDE_PLAN = """Ahora escribe el plan para resolverla, en JSON, sin nada más al final:
{{"pasos": [{{"paso": "qué hacer", "archivo": "dónde", "comprobacion": "qué prueba lo demuestra"}}]}}
Cada paso debe decir cómo se comprobará que está bien hecho."""


def leer_json(respuesta, clave):
    """El último objeto JSON de una respuesta que tenga la clave dada, o None."""
    for trozo in reversed(re.findall(r"\{.*\}", respuesta, re.S)):
        try:
            datos = json.loads(trozo, strict=False)
        except ValueError:
            continue
        if isinstance(datos, dict) and clave in datos:
            return datos
    return None


def comprobar_evidencias(marco, espacio, leidas=None):
    """Los hechos del marco cuya evidencia no existe: un archivo que no está o una
    línea que no tiene. Con «leidas», además, la línea citada tiene que haberse leído
    con leer_archivo durante la fase previa."""
    malas = []
    for hecho in marco.get("hechos", []):
        m = re.match(r"\s*([\w./-]+\.\w+)(?::(\d+))?", str(hecho.get("evidencia", "")))
        archivo = espacio.ruta(m.group(1)) if m else None
        if not m or not archivo.is_file():
            malas.append(f"«{hecho.get('hecho', '')}»: no existe {hecho.get('evidencia')}")
        elif m.group(2) and int(m.group(2)) > len(
                archivo.read_text(encoding="utf-8").splitlines()):
            malas.append(f"«{hecho.get('hecho', '')}»: {m.group(1)} no tiene la línea "
                         f"{m.group(2)}")
        elif leidas is not None and (not m.group(2) or int(m.group(2)) not in
                                     leidas.get(m.group(1), set())):
            malas.append(f"«{hecho.get('hecho', '')}»: no has leído la línea que citas "
                         f"({hecho.get('evidencia')}); léela antes de citarla")
    return malas


def anotar_evidencias(marco, espacio):
    """El marco con el texto real de cada línea citada, para que quien lo lea vea si el
    hecho y su evidencia dicen lo mismo."""
    anotado = dict(marco, hechos=[])
    for hecho in marco.get("hechos", []):
        m = re.match(r"\s*([\w./-]+\.\w+):(\d+)", str(hecho.get("evidencia", "")))
        linea = ""
        if m and espacio.ruta(m.group(1)).is_file():
            lineas = espacio.ruta(m.group(1)).read_text(encoding="utf-8").splitlines()
            if 0 < int(m.group(2)) <= len(lineas):
                linea = lineas[int(m.group(2)) - 1].strip()[:120]
        anotado["hechos"].append(dict(hecho, texto_de_la_linea=linea))
    return anotado


def conversar(tok, modelo, mensajes, herramientas, max_pasos, clave, ver=False):
    """Una conversación con herramientas de lectura que acaba con un JSON con «clave».
    Devuelve el JSON (o None) y los pasos."""
    for pasos in range(1, max_pasos + 1):
        texto, recortados = recortar_conversacion(tok, mensajes)
        if recortados is None:
            return None, pasos
        respuesta, _, _ = generar(tok, modelo, texto)
        llamadas = leer_llamadas(respuesta)
        if not llamadas:
            mensajes.append({"role": "assistant", "content": respuesta})
            return leer_json(respuesta, clave), pasos
        mensajes.append({"role": "assistant", "content": respuesta.split("<tool_call>")[0],
                         "tool_calls": [{"type": "function", "function": l}
                                        for l in llamadas if l["name"]]})
        for llamada in llamadas:
            resultado = (herramientas.llamar(llamada["name"], llamada["arguments"])
                         if llamada["name"] else {"error": "Llamada no válida."})
            if ver:
                print(f"  [marco {pasos}] {llamada['name']}"
                      f"({json.dumps(llamada['arguments'], ensure_ascii=False)[:80]})")
            mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                             "content": json.dumps(resultado, ensure_ascii=False)})
    return None, max_pasos


def enmarcar(tok, modelo, tarea, espacio, con_plan, estricto=False, ver=False):
    """La fase previa: el agente explora sin cambiar nada y escribe el marco de la
    tarea; el programa comprueba las evidencias y, si alguna no existe, se lo devuelve
    una vez. Con «con_plan», escribe además un plan con una comprobación por paso. Con
    «estricto», cada evidencia tiene que ser una línea leída en esta fase, y el marco
    que recibe el agente lleva el texto real de cada línea citada. Devuelve el texto
    que se añade a la petición y lo que ha pasado."""
    alcance = tarea["alcance"]
    mensajes = [{"role": "system", "content": SISTEMA + "\n\n" + AGENTS_REGLAS},
                {"role": "user", "content": PIDE_MARCO.format(
                    peticion=tarea["peticion"], contrato=contrato_texto(alcance))}]
    lectura = HerramientasDeLectura(hr.Herramientas(espacio))
    marco, pasos = conversar(tok, modelo, mensajes, lectura, MAX_PASOS_MARCO,
                             "objetivo", ver)
    leidas = lectura.leidas if estricto else None
    malas = comprobar_evidencias(marco, espacio, leidas) if marco else []
    if malas:
        mensajes.append({"role": "user", "content":
                         "Estas evidencias no existen: " + "; ".join(malas)
                         + ". Corrige el marco y vuelve a escribirlo en JSON."})
        corregido, mas = conversar(tok, modelo, mensajes, lectura, 4, "objetivo", ver)
        pasos += mas
        marco = corregido or marco
        malas = comprobar_evidencias(marco, espacio, leidas) if marco else malas
    plan = None
    if con_plan and marco:
        mensajes.append({"role": "user", "content": PIDE_PLAN})
        plan, mas = conversar(tok, modelo, mensajes, lectura, 2, "pasos", ver)
        pasos += mas
    partes = []
    if marco:
        visible = anotar_evidencias(marco, espacio) if estricto else marco
        partes.append("Marco de la tarea (escrito antes de empezar):\n"
                      + json.dumps(visible, ensure_ascii=False, indent=1)[:2500])
    if plan:
        partes.append("Plan:\n" + json.dumps(plan, ensure_ascii=False, indent=1)[:1500])
    return "\n\n".join(partes), {"marco": marco, "plan": plan, "pasos": pasos,
                                 "evidencias_malas": malas}

In [ ]:
marco = {"hechos": [{"hecho": "el umbral está en config", "evidencia": "config.py:11"},
                    {"hecho": "hay un módulo de cupones", "evidencia": "cupones.py:3"}]}
with hr.Espacio() as espacio:
    print(comprobar_evidencias(marco, espacio))

## 7. El agente con el marco y el plan

La refactorización de los números mágicos (T19), sin marco y con marco y plan.

In [ ]:
PIDE_REPLANIFICAR = ("Todavía no lo has terminado. Antes de dejarlo, vuelve a entender la "
                     "tarea con el repositorio tal como está ahora. Este es el marco y el "
                     "plan que acabas de escribir; síguelo paso a paso y responde TERMINADO "
                     "cuando sus comprobaciones pasen.\n\n")


def agente(tok, modelo, tarea, marco=False, plan=False, estricto=False,
           replanificar=False, revisor=None, max_pasos=MAX_PASOS, ver=False):
    """El agente del capítulo 5 (contrato, retroalimentación y puerta de verificación),
    con el revisor del capítulo 6 si se pide, y con una fase previa opcional: «marco»,
    el agente enmarca la tarea antes de trabajar, y «plan», además la planifica. El
    marco y el plan se añaden a la petición. Con «replanificar», el agente empieza sin
    marco y solo lo escribe, con su plan, la primera vez que va a rendirse sin haber
    terminado; luego sigue con los pasos que le queden. Devuelve la fila de resultados."""
    inicio = time.time()
    alcance = tarea["alcance"]
    peticion = tarea["peticion"] + "\n\n" + contrato_texto(alcance)
    fase = {"marco": None, "plan": None, "pasos": 0, "evidencias_malas": []}
    with hr.Espacio() as espacio:
        if marco:
            previo, fase = enmarcar(tok, modelo, tarea, espacio, con_plan=plan,
                                    estricto=estricto, ver=ver)
            if previo:
                peticion += "\n\n" + previo
        mensajes = [{"role": "system", "content": SISTEMA + "\n\n" + AGENTS_REGLAS},
                    {"role": "user", "content": peticion}]
        final, pasos, entrada, salida, rechazos = None, 0, 0, 0, 0
        revisiones = []
        base = hr.Herramientas(espacio)
        herramientas = HerramientasConRetro(base, tarea, alcance)
        for pasos in range(1, max_pasos + 1):
            texto, recortados = recortar_conversacion(tok, mensajes)
            if recortados is None:
                break
            respuesta, n_entrada, n_salida = generar(tok, modelo, texto)
            entrada, salida = entrada + n_entrada, salida + n_salida
            llamadas = leer_llamadas(respuesta)
            if not llamadas:
                mensajes.append({"role": "assistant", "content": respuesta})
                if ver:
                    print(f"[{pasos}] {respuesta}")
                if dice_terminado(respuesta) and rechazos < MAX_RECHAZOS:
                    cambios = hr.diferencias(espacio.origen, espacio.carpeta)
                    rotas = comprobar_reglas(tarea, base.registro, cambios, respuesta,
                                             exigir_prueba=not prohibe_pruebas(alcance),
                                             permite_docs=herramientas.permite_docs())
                    rotas.update(comprobar_contrato(alcance, cambios, respuesta))
                    rotas.pop("R7", None)
                    falta = puerta_de_verificacion(espacio, alcance)
                    if falta:
                        rotas["Verificación"] = "; ".join(falta)
                    if rotas:
                        rechazos += 1
                        mensajes.append({"role": "user", "content":
                                         "Todavía no está terminado: "
                                         + "; ".join(f"{r}: {t}" for r, t in
                                                     sorted(rotas.items()))
                                         + ". Sigue trabajando."})
                        continue
                if (replanificar and not fase["marco"] and not dice_terminado(respuesta)
                        and pasos < max_pasos):
                    previo, fase = enmarcar(tok, modelo, tarea, espacio, con_plan=True,
                                            ver=ver)
                    mensajes.append({"role": "user", "content": PIDE_REPLANIFICAR
                                     + (previo or "No has conseguido escribir el marco.")})
                    continue
                if revisor and dice_terminado(respuesta):
                    if len(revisiones) == MAX_REVISIONES:   # ya lo rechazó dos veces
                        final = ESCALADO
                        break
                    revision = revisar(tok, modelo, tarea, alcance, espacio, respuesta,
                                       rubrica=revisor == "rúbrica", ver=ver)
                    revisiones.append(revision)
                    if not revision["aprobado"]:
                        mensajes.append({"role": "user", "content":
                                         "El revisor no aprueba el cambio: "
                                         + comentarios_para_el_autor(revision["veredicto"])
                                         + ". Corrígelo y vuelve a responder TERMINADO."})
                        continue
                final = respuesta
                break
            pensamiento = respuesta.split("<tool_call>")[0].strip()
            mensajes.append({"role": "assistant", "content": pensamiento,
                             "tool_calls": [{"type": "function", "function": l}
                                            for l in llamadas if l["name"]]})
            for llamada in llamadas:
                if llamada["name"] is None:
                    resultado = {"error": "No se entiende la llamada a la herramienta."}
                else:
                    resultado = herramientas.llamar(llamada["name"], llamada["arguments"])
                if ver:
                    print(f"[{pasos}] {llamada['name']}"
                          f"({json.dumps(llamada['arguments'], ensure_ascii=False)[:90]})"
                          f" -> {json.dumps(resultado, ensure_ascii=False)[:110]}")
                mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                                 "content": json.dumps(resultado, ensure_ascii=False)})
        evaluacion = hr.evaluar_intento(espacio, tarea)
    terminado = final is not None and dice_terminado(final)
    return {"tarea": tarea["id"], "tipo": tarea["meta"]["tipo"],
            "resuelta": evaluacion["resuelta"], "dice_terminado": terminado,
            "terminado_falso": terminado and not evaluacion["resuelta"],
            "visibles": evaluacion["visibles"], "ocultas": evaluacion["ocultas"],
            "fuera_de_alcance": evaluacion["fuera_de_alcance"], "rechazos": rechazos,
            "revisiones": revisiones, "fase_previa": fase,
            "pasos": pasos, "tokens_entrada": entrada, "tokens_salida": salida,
            "segundos": round(time.time() - inicio, 1),
            "respuesta": final or "(sin respuesta final)",
            "llamadas": [[r["herramienta"], r["argumentos"].get("ruta", ""), r["ok"]]
                         for r in base.registro],
            "diff": evaluacion["diff"]}


VARIANTES = {   # todas con el revisor con rúbrica del capítulo 6
    "sin marco": {"revisor": "rúbrica"},
    "con marco": {"marco": True, "revisor": "rúbrica"},
    "marco y plan": {"marco": True, "plan": True, "revisor": "rúbrica"},
    "marco con evidencias": {"marco": True, "plan": True, "estricto": True,
                             "revisor": "rúbrica"},
    "marco si se atasca": {"replanificar": True, "revisor": "rúbrica"},
}

In [ ]:
for nombre in ("sin marco", "marco y plan"):
    print(f"--- {nombre}")
    fila = agente(tok, modelo, hr.leer_tarea("T19"), ver=True, **VARIANTES[nombre])
    print("Resuelta:", fila["resuelta"], "· Respuesta:", fila["respuesta"][:120])

## 8. Las veinticuatro tareas, una ejecución voraz por variante

Las cinco variantes tardan unas catorce horas en una GPU de escritorio. Para una primera prueba, pasa `tareas=BANCO[:4]` a `probar`.

In [ ]:
TAREAS_MEDIAS = [f"T{n}" for n in range(21, 31)]   # el banco ampliado del capítulo 5


BANCO = TAREAS + TAREAS_MEDIAS


def resumen(filas):
    """Los totales de una variante, con la fase previa."""
    con_marco = [f for f in filas if f["fase_previa"]["marco"]]
    return {"tareas": len(filas),
            "resueltas": sum(f["resuelta"] for f in filas),
            "dice_terminado": sum(f["dice_terminado"] for f in filas),
            "terminado_falso": sum(f["terminado_falso"] for f in filas),
            "pruebas_en_rojo": sum(not f["visibles"] for f in filas),
            "con_marco": len(con_marco),
            "con_plan": sum(bool(f["fase_previa"]["plan"]) for f in filas),
            "evidencias_malas": sum(len(f["fase_previa"]["evidencias_malas"]) for f in filas),
            "pasos": sum(f["pasos"] for f in filas),
            "pasos_previos": sum(f["fase_previa"]["pasos"] for f in filas),
            "tokens_entrada": sum(f["tokens_entrada"] for f in filas),
            "segundos": round(sum(f["segundos"] for f in filas), 1)}


def probar(tok, modelo, opciones, tareas=BANCO, semilla=None):
    """Una variante con todas las tareas: sus filas y su resumen. Sin semilla, con
    decodificación voraz; con semilla, con muestreo, y cada tarea con su propia semilla
    para que la ejecución se pueda repetir."""
    decodificacion.clear()
    decodificacion.update(VORAZ if semilla is None else MUESTREO)
    filas = []
    for numero, identificador in enumerate(tareas):
        if semilla is not None:
            torch.manual_seed(1000 * semilla + numero)
        fila = agente(tok, modelo, hr.leer_tarea(identificador), **opciones)
        filas.append(fila)
        print(f"  {identificador} {'resuelta' if fila['resuelta'] else 'no      '}"
              f"  {'dice terminado' if fila['dice_terminado'] else 'no lo dice    '}"
              f"  marco={'sí' if fila['fase_previa']['marco'] else 'no'}"
              f"  previos={fila['fase_previa']['pasos']:2d}"
              f"  pasos={fila['pasos']:2d}  {fila['segundos']:6.1f} s", flush=True)
    return {"filas": filas, "resumen": resumen(filas)}

In [ ]:
resultados = {}
for nombre, opciones in VARIANTES.items():
    print(f"== {nombre}")
    resultados[nombre] = probar(tok, modelo, opciones)
    print(resultados[nombre]["resumen"])

## 9. Medir bien: tres ejecuciones con muestreo

Cada ejecución, con su semilla, se puede repetir exactamente. Las seis tardan unas catorce horas; para probar, usa pocas tareas, por ejemplo `tareas=TAREAS[:5]`.

In [ ]:
repeticiones = {}
for semilla in (1, 2, 3):
    for nombre in ("sin marco", "marco y plan"):
        r = probar(tok, modelo, VARIANTES[nombre], semilla=semilla)
        repeticiones[f"{nombre} | semilla {semilla}"] = r
        print(nombre, semilla, r["resumen"]["resueltas"], "resueltas")

## Ejercicios

Los archivos de resultados que citan los ejercicios están en `datos/resultados/capitulo_7/`, que `preparar` descarga con los datos.

1. ¿Qué diferencia hay entre el marco y el plan? Pon un ejemplo de una tarea en la que baste con el marco y otra en la que haga falta el plan. *(Sin GPU.)*

2. Lee los marcos de la variante con marco y plan en tres tareas. ¿Cuántas evidencias dicen de verdad lo que el hecho afirma? Compruébalo abriendo cada línea. *(Sin GPU.)*

3. Ejecuta el agente sin marco con decodificación voraz en la tarea del IVA (T04), y después añade una frase cualquiera al final de la petición, por ejemplo «Gracias». ¿Cambia el resultado? Los resultados también están en `resultados_ejercicios.json`. *(GPU, unos 20 minutos, o sin GPU con los resultados guardados.)*

4. Calcula, con `resultados_repeticiones.json`, la media y el rango de tareas resueltas de cada variante solo en las catorce tareas originales y solo en las diez medianas. *(Sin GPU.)*

5. Con las tres ejecuciones de cada variante, calcula la diferencia por semilla y cuánto varía. ¿Cuántas ejecuciones harían falta para que la diferencia de medias fuera el doble de su error típico? Si quieres añadir una semilla, hazlo solo con las cinco tareas en las que más difieren las variantes (T01, T03, T05, T17 y T19). *(Sin GPU; con una semilla más en esas cinco tareas, GPU, unos 45 minutos.)*

6. El agente con plan cita evidencias que existen pero no dicen lo que afirma. Escribe una comprobación que lo detecte en algún caso sin leer la línea con un modelo, por ejemplo buscando en la línea el nombre de la constante que cita el hecho. *(Sin GPU.)*

7. ¿Por qué la variante que replanifica al atascarse gasta el doble? Busca en sus resultados una tarea en la que el plan llegue tarde y cuenta los pasos antes y después. *(Sin GPU.)*

8. Pide el plan sin el marco: el agente escribe solo los pasos, sin explorar antes. ¿Qué pierde el plan? *(GPU, unos 30 minutos.)* Basta con las tareas de varias piezas, como T03 y T19.

9. Busca en tu entorno de trabajo una decisión que se haya tomado con una sola medida: un experimento, una prueba de rendimiento, una comparación de herramientas. ¿Cuántas repeticiones habría necesitado? *(Sin GPU.)*

10. **Proyecto.** Elige dos formas de pedir una tarea a un agente de programación que uses, comercial o el del libro, y ejecuta cada una tres veces sobre cinco tareas tuyas. Presenta el resultado con la media, el rango y las tareas que salen «a veces».

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Plan-and-Solve Prompting](https://arxiv.org/abs/2305.04091) (Wang y colaboradores, 2023). [AI Agents That Matter](https://arxiv.org/abs/2407.01502) (Kapoor y colaboradores, 2024). La tienda Ciclos Norte, su programa y sus tareas son ficticios. Código y experimentos propios.